# **Import Library**

In [1]:
import pandas as pd # Untuk menganalisa data (Membaca, Memanipulasi, Membersihkan)

import joblib # Untuk membuat model dapat digunakan kembali

# Library Sikit Learn
from sklearn.model_selection import train_test_split # Untuk membagi dataset menjadi dua bagian (Traning & Testing)
from sklearn.feature_extraction.text import TfidfVectorizer # Untuk mengubah kalimat menjadi pecahan kata
from sklearn.pipeline import FeatureUnion # Untuk mengubah pecahan kata menjadi satu kalimat gabungan
from sklearn.ensemble import RandomForestClassifier # Metode pengambilan keputusan model
from sklearn.metrics import classification_report, accuracy_score # Untuk menampilkan rangkuman score dalam bentuk tabel


# **Memuat Dataset (Load)**

In [3]:
# Memuat dataset .csv
df = pd.read_csv("dataset/judol.csv")

In [4]:
df.head()

,video_id,title,channel_name,tanggal,author,komentar,label,komentar_clean,predicted_label
0,f8UEkmYXlzA,SKAKMAT AHOK,Pandji Pragiwaksono,1.745.412.234.123.970,TerranceNoelle-o9i,Makin yakin abis baca review lain tentang ✌✌𝐒𝐆...,1,makin yakin abis baca review lain tentang 𝐒𝐆𝐈𝟖𝟖 .,0
1,XI8K0-_kbHc,GAK NYANGKA BISA BEGINI! PENGENDARA DIJALAN SA...,Jejelogy,1.739.601.493.342.000,deraatvexplorerriders.2113,Paling suka model H2 😍🔥,0,suka model h2,0
2,nZoNbiwP2ZE,Akhirnya Selesai Subaru Crosstrek Family Drift...,Garasi Drift,1.739.772.479.808.090,risqokurniadi7208,Mobilnya udah hancur 🥺,0,mobilnya udah hancur,0
3,QpXcKzQInXg,Review Mobil Drift Seharga Super Car | BRZ V8 ...,Garasi Drift,1.738.825.556.100.780,LorrianeDotson,░𝙈𝘼𝙉𝙐𝙏88░benar2 bikin aku jadi sultan,1,░𝙈𝘼𝙉𝙐𝙏88░benar2 bikin sultan,1
4,nZoNbiwP2ZE,Akhirnya Selesai Subaru Crosstrek Family Drift...,Garasi Drift,1.739.858.865.953.920,Elpoco7365,Semoga lekas recover mobilnya mas Dipo,0,semoga lekas recover mobilnya mas dipo,0


# **Pra-Pemrosesan Data (Preprocessing)**

In [5]:
# Isi data yang kosong (NaN) dengan string kosong agar tidak error
df['komentar_clean'] = df['komentar_clean'].fillna('').astype(str).str.lower()

# Membuat stop_words menggunakan TF-IDF agar kata seperti 'di', 'ke', 'ini' dihilangkan
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), max_features=5000)
char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=5000)

# Gabungkan dua metode ekstraksi menjadi satu kesatuan data
vectorizer = FeatureUnion([
    ('word_proc', word_vectorizer),
    ('char_proc', char_vectorizer)
])

X = vectorizer.fit_transform(df['komentar_clean']) # Mengubah kolom komentar_clean menjadi matrix angka
y = df['label'] # Target/Jawaban yang akan ditebak oleh model


# Simpan model agar dapat digunakan kembali untuk proses selanjutnya
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

# **Data Splitting**

In [6]:
# Menggunakan train_test_split() untuk melakukan pembagian dataset
#  Gunakan 'stratify=y' agar proporsi kelas di train/test set sama.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# **Membangun Model**


In [7]:
# Menggunakan RandomForestClassifier untuk pengambilan keputusan model
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [8]:
joblib.dump(rf_model, 'model_final.pkl')

['model_final.pkl']

# **Evaluasi Model**

In [9]:
# Model melakukan prediksi dari data test
y_pred = rf_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))

Accuracy: 97.41%
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1204
           1       0.98      0.96      0.97       842

    accuracy                           0.97      2046
   macro avg       0.97      0.97      0.97      2046
weighted avg       0.97      0.97      0.97      2046



In [10]:
def improved_gatekeeper(text):
    # Tambahkan kata-kata dari dataset yang sering membuat model ragu
    blacklist = ['sgi88', 'manut88', 'alexis17', 'weton88', 'jp paus', 'slot gacor', 'scatter']

    # Cek Blacklist
    if any(word in text.lower() for word in blacklist):
        return {
            "status": "JUDI ONLINE",
            "score": 1.0,
            "method": "HARD_RULE_MATCH"
        }

    # Cek AI Lokal
    model = joblib.load('model_final.pkl')
    v = joblib.load('tfidf_vectorizer.pkl')


    vector = v.transform([text.lower()])
    probs = model.predict_proba(vector)[0]
    prediction = model.predict(vector)[0]
    confidence = max(probs)

    # Threshold Logic
    if confidence < 0.85:
        return {
            "status": "RAGU-RAGU",
            "score": float(confidence),
            "method": "TRIGGER_AZURE_AI"
        }

    label = "JUDI/FRAUD" if prediction == 1 else "AMAN"
    return {
        "status": label,
        "score": float(confidence),
        "method": "LOCAL_AI_DECISION"
    }

# Lakukan pengujian dengan Study Case
test_case_1 = "Review GadgetIn emang paling mantap"
test_case_2 = "SGI88 beneran bikin jp paus"
test_case_3 = "kurang berapa menit lagi ni"

for tc in [test_case_1, test_case_2, test_case_3]:
    res = improved_gatekeeper(tc)
    print(f"Input: {tc}\nResult: {res['status']} | Score: {res['score']:.2%}")

Input: Review GadgetIn emang paling mantap
Result: AMAN | Score: 86.00%
Input: SGI88 beneran bikin jp paus
Result: JUDI ONLINE | Score: 100.00%
Input: kurang berapa menit lagi ni
Result: AMAN | Score: 99.00%
